# Red de transferencias de tokens Ethereum

**Proyecto final — Estructura de Datos**
**Universidad Industrial de Santander**

**Profesor:** Jonnathan Alfredo Ramos Chaux
**Equipo:** Jhon Arciniegas, Pablo Álvarez y Emerson Ballesteros (líder)

[Abrir este notebook en Google Colab](https://colab.research.google.com/github/arcibyte/ethereum-exchange/blob/main/notebooks/Proyecto_Teoria_Grafos.ipynb)

---

## ¿De qué trata el proyecto?

Analizamos un **grafo** construido a partir de transferencias de un token
ERC-20 de Ethereum. Las direcciones son los **nodos** y las transferencias
son las **aristas** (dirigidas y con peso = cantidad transferida).

A partir de ese grafo resolvemos **4 tareas**, una por integrante:

| Tarea | Pregunta | Algoritmo | Responsable |
|-------|----------|-----------|-------------|
| 1 | ¿Cuál es el camino más barato entre dos direcciones? | Bellman-Ford | Pablo Álvarez |
| 2 | ¿Qué nodos son los más importantes? | Centralidad + PageRank | Jhon Arciniegas |
| 3 | ¿Qué grupos de nodos interactúan más entre sí? | Louvain (comunidades) | Emerson Ballesteros |
| 4 | ¿Cuál es la red mínima que conecta a todos? | Kruskal (MST) | Grupal |

**Reglas del proyecto:**
- No se permite usar el algoritmo de **Dijkstra**.
- Todo se implementa desde cero con Python (listas y diccionarios).

> **Celda vs. módulo:** en Colab, cada cuadro de código es una **celda**.
> La implementación de cada algoritmo vive en un **módulo** `.py` dentro de
> `src/`. Las celdas solo importan y usan esos módulos.

## Análisis en dos capas

1. **Los cuatro algoritmos** se ejecutan sobre un subgrafo conectado de pocas
   miles de aristas, para que el Python puro termine en Colab.
2. **El dataset completo (38,9 M de transferencias)** se analiza al final con
   DuckDB, directamente sobre el Parquet de ~0,8 GB.

> **Cómo ejecutar:** corre las celdas en orden. La primera celda de
> configuración clona el repositorio. La **celda de carga única** de la
> Sección 1 se ejecuta una vez; las cuatro tareas reutilizan el mismo objeto
> `grafo` y no vuelven a leer los datos.

## Configuración

Esta celda deja el entorno listo **tanto en Google Colab como en local**:

- En **Colab**, clona el repositorio desde GitHub y se ubica en su raíz.
- En **local**, detecta la raíz del repositorio automáticamente.

No necesitas cambiar rutas a mano: la celda lo hace por ti.


In [ ]:
# ============ CONFIGURACIÓN DEL ENTORNO (Colab + local) ============
import os
import sys
import subprocess

REPO_URL = "https://github.com/arcibyte/ethereum-exchange"
REPO_BRANCH = "main"  # para probar otra rama: "feat/celda-carga-unica"
REPO_DIR = "ethereum-exchange"
EN_COLAB = "google.colab" in sys.modules

# Si editas un módulo .py, Colab lo vuelve a importar al ejecutar la celda.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass


def _raiz_del_repo():
    """Sube por las carpetas hasta encontrar la raíz (donde está src/grafo.py)."""
    actual = os.getcwd()
    while True:
        if os.path.isfile(os.path.join(actual, "src", "grafo.py")):
            return actual
        padre = os.path.dirname(actual)
        if padre == actual:
            return None
        actual = padre


if EN_COLAB:
    # Si ya estamos dentro del repo (por una ejecución anterior), no clonamos.
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            print("Clonando el repositorio...")
            subprocess.run(
                ["git", "clone", "--branch", REPO_BRANCH, REPO_URL],
                check=True,
            )
        os.chdir(REPO_DIR)
else:
    raiz = _raiz_del_repo()
    if raiz is not None:
        os.chdir(raiz)

# La raíz del repo debe estar en sys.path para poder importar `src`.
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print("Directorio de trabajo:", os.getcwd())

## 1. Carga única del dataset

Esta es la **única celda que lee datos para los cuatro algoritmos**. Deja un
objeto `grafo` con:

- `grafo.aristas`: lista de `(origen, destino, valor)`.
- `grafo.nodos`: todas las direcciones.
- `grafo.etiquetas`: nombres legibles de exchanges.

Cada módulo recibe `grafo.aristas` y construye **su propia estructura**. Ningún
módulo vuelve a abrir el archivo.

Elige la fuente con `FUENTE`:

| `FUENTE` | Qué carga | Para qué sirve |
|----------|-----------|----------------|
| `"sample"` | Muestra mínima del repo | Demo rápida, corre al instante |
| `"muestra"` | Muestra conectada de USDC (~2.000 aristas) | Análisis con datos reales |
| `"parquet"` | Muestra aleatoria del Parquet completo | Prueba con el dataset completo |

> Los algoritmos son Python puro y Bellman-Ford es $O(V \cdot E)$: por eso el
> grafo de las tareas usa un **subgrafo conectado** de pocas miles de aristas.
> El dataset completo se analiza aparte con DuckDB al final del notebook.

In [ ]:
# ============ CARGA ÚNICA DEL DATASET ============
# Esta celda construye UNA sola vez el objeto `grafo` que usan las 4 tareas.

import os

from src.grafo import cargar_grafo

# --- Elige la fuente del grafo ---
# "sample"  : muestra mínima del repo (12 direcciones, corre al instante)
# "muestra" : CSV preparado con la muestra conectada de USDC (se carga completo)
# "parquet" : muestra aleatoria del Parquet real (~38M de transferencias)
FUENTE = "muestra"

RUTA_SAMPLE = "data/sample/transferencias.csv"
RUTA_MUESTRA = "data/muestra/transferencias_2k.csv"
RUTA_PARQUET = "/content/transferencias.parquet"
RUTA_PARQUET_LOCAL = "data/procesado/transferencias.parquet"
RUTA_PARQUET_DRIVE = "/content/drive/MyDrive/ED2/transferencias.parquet"
RUTA_ETIQUETAS_SAMPLE = "data/sample/etiquetas.csv"
RUTA_ETIQUETAS_MUESTRA = "data/muestra/etiquetas.csv"

# Aristas que se cargan cuando la fuente es un Parquet o un CSV grande.
# Para Louvain simple y Bellman-Ford en Python puro, 2.000 es un tamaño seguro.
MAX_ARISTAS = 2_000
SEMILLA = 42
CONSERVAR_COMPONENTE_MAYOR = True

# En Colab, si el Parquet está en Drive, descomenta:
# from google.colab import drive
# drive.mount("/content/drive")


def _primera_existente(*rutas):
    for ruta in rutas:
        if ruta and os.path.isfile(ruta):
            return ruta
    return None


if FUENTE == "sample":
    ruta = RUTA_SAMPLE
    opciones = {}
elif FUENTE == "muestra":
    ruta = RUTA_MUESTRA
    opciones = {}   # el CSV ya es la muestra conectada preparada
elif FUENTE == "parquet":
    ruta = _primera_existente(RUTA_PARQUET_LOCAL, RUTA_PARQUET, RUTA_PARQUET_DRIVE)
    if ruta is None:
        raise FileNotFoundError(
            "No encontré el Parquet. Genera uno con "
            "notebooks/Preparar_Dataset_Colab.ipynb o ajusta las rutas."
        )
    opciones = {"muestra": MAX_ARISTAS, "semilla": SEMILLA}
else:
    raise ValueError('FUENTE debe ser "sample", "muestra" o "parquet".')

# ---- LA CARGA OCURRE UNA SOLA VEZ ----
if FUENTE == "sample":
    etiquetas = RUTA_ETIQUETAS_SAMPLE
else:
    etiquetas = _primera_existente(RUTA_ETIQUETAS_MUESTRA, RUTA_ETIQUETAS_SAMPLE)
grafo = cargar_grafo(ruta, etiquetas=etiquetas, **opciones)

if CONSERVAR_COMPONENTE_MAYOR and FUENTE != "sample":
    grafo.conservar_componente_mayor()

print("Fuente:", ruta)
print("Resumen del grafo:")
for clave, valor in grafo.resumen().items():
    print(f"  {clave}: {valor}")
print("Aristas disponibles para los módulos:", len(grafo.aristas))

### 1.1 Vista rápida del grafo cargado

El resto del notebook usa el mismo objeto `grafo`. Esta celda solo lo
describe; no vuelve a leer el archivo.

In [ ]:
# Vista rápida: distribución de grados del grafo ya cargado.
import matplotlib.pyplot as plt
from collections import Counter

salidas = Counter(origen for origen, _, _ in grafo.aristas)
entradas = Counter(destino for _, destino, _ in grafo.aristas)
grados = {nodo: salidas[nodo] + entradas[nodo] for nodo in grafo.nodos}

plt.figure(figsize=(6, 4))
plt.hist(list(grados.values()), bins=30, color="steelblue", edgecolor="white")
plt.xlabel("Grado")
plt.ylabel("Frecuencia")
plt.title("Distribución de grados del grafo de trabajo")
plt.show()

print("Nodos:", len(grafo.nodos), "| Aristas:", len(grafo.aristas))

## Tarea 1 — Caminos de costo mínimo (Bellman-Ford)

**Responsable:** Pablo Álvarez

**Problema:** queremos saber cuál es la ruta más barata (menor cantidad
transferida) para ir de una dirección a otra.

**Equivalencia matemática:** camino de costo mínimo en un grafo dirigido y
ponderado. El costo de un camino es la suma de los pesos de sus aristas.

**Alternativas consideradas:**

| Algoritmo | ¿Por qué sí / no? |
|-----------|-------------------|
| Dijkstra | Prohibido por las reglas del proyecto |
| BFS | Solo sirve cuando todas las aristas pesan lo mismo |
| **Bellman-Ford** | Funciona con pesos reales y detecta ciclos negativos |

**Elegimos Bellman-Ford.** Su complejidad es $O(|V| \cdot |E|)$, aceptable
para grafos de tamaño moderado.


In [ ]:
# Tarea 1 — usa el mismo `grafo` cargado una sola vez.
from collections import Counter

from src.bellman_ford import bellman_ford_desde_aristas, reconstruir_camino

# Elegimos como origen un exchange etiquetado con muchas salidas (si hay).
salidas = Counter(origen for origen, _, _ in grafo.aristas)
etiquetados = sorted(
    (nodo for nodo in salidas if nodo in grafo.etiquetas),
    key=salidas.get, reverse=True,
)
origen = etiquetados[0] if etiquetados else max(salidas, key=salidas.get)

distancias, previos = bellman_ford_desde_aristas(grafo.aristas, origen)

alcanzables = sorted(
    ((d, nodo) for nodo, d in distancias.items()
     if d != float("inf") and nodo != origen)
)
inalcanzables = sum(1 for d in distancias.values() if d == float("inf"))

print(f"Origen: {grafo.nombre(origen)}")
print(f"Nodos alcanzables: {len(distancias) - inalcanzables - 1:,}")
print(f"Nodos inalcanzables: {inalcanzables:,}\n")

print("Los 10 destinos más baratos:")
for d, destino in alcanzables[:10]:
    camino = reconstruir_camino(previos, origen, destino)
    nombres = " -> ".join(grafo.nombre(n) for n in camino)
    print(f"  {grafo.nombre(destino):<16} : costo {d:>14,.1f}   ({nombres})")

In [ ]:
# Experimento T1: caminos desde los 5 nodos con más salidas.
# Sirve para ver que Bellman-Ford funciona igual desde cualquier origen.
print(f"{'Origen':<14} {'Nodos':>6} {'Alcanzables':>12} {'Costo máximo':>14}")
print("-" * 52)

candidatos = sorted(
    (nodo for nodo in salidas if nodo in grafo.etiquetas),
    key=salidas.get, reverse=True,
)
if len(candidatos) < 5:
    candidatos += [n for n, _ in salidas.most_common(10) if n not in candidatos]

for origen_exp in candidatos[:5]:
    dist, _ = bellman_ford_desde_aristas(grafo.aristas, origen_exp)
    alcanzables = [d for d in dist.values() if d != float("inf")]
    costo_max = max(alcanzables) if alcanzables else 0.0
    print(f"{grafo.nombre(origen_exp):<14} {len(dist):>6} "
          f"{len(alcanzables):>12} {costo_max:>14.1f}")

## Tarea 2 — Centralidad e influencia

**Responsable:** Jhon Arciniegas

**Problema:** ¿cuáles son los nodos más importantes de la red?

**Equivalencia matemática:** dos medidas de centralidad:

- **Grado:** $C(v) = \text{grado}(v) / (|V| - 1)$
- **PageRank:** $PR(v) = \frac{1-d}{N} + d \sum_{u \to v} \frac{PR(u)}{\text{grado_salida}(u)}$

**Alternativas:** la centralidad de cercanía y la de intermediación también
miden importancia, pero requieren calcular caminos mínimos repetidamente. Para
un primer análisis bastan el grado y PageRank.


In [ ]:
# Tarea 2 — usa el mismo `grafo` cargado una sola vez.
from src.centralidad import (
    centralidad_grado_desde_aristas,
    pagerank_desde_aristas,
    top_n,
)

cent_grado = centralidad_grado_desde_aristas(grafo.aristas)
pr = pagerank_desde_aristas(grafo.aristas)

print("Top 10 por centralidad de grado:")
for addr, score, nombre in top_n(cent_grado, n=10, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<16} : {score:.4f}")

print("\nTop 10 por PageRank:")
for addr, score, nombre in top_n(pr, n=10, etiquetas=grafo.etiquetas):
    print(f"  {nombre:<16} : {score:.4f}")

# Gráfica del top 10 por PageRank.
top = top_n(pr, n=10, etiquetas=grafo.etiquetas)
nombres = [nombre[:16] for _, _, nombre in top]
valores = [score for _, score, _ in top]

plt.figure(figsize=(8, 4))
plt.barh(nombres[::-1], valores[::-1], color="mediumpurple")
plt.xlabel("PageRank")
plt.title("Top 10 nodos por PageRank")
plt.tight_layout()
plt.show()

In [ ]:
# Experimento T2: ¿coinciden el top por grado y el top por PageRank?
top_grado = {addr for addr, _, _ in top_n(cent_grado, n=10)}
top_pagerank = {addr for addr, _, _ in top_n(pr, n=10)}

print("Nodos en ambos top-10:", len(top_grado & top_pagerank))
print("Solo en grado      :", len(top_grado - top_pagerank))
print("Solo en PageRank   :", len(top_pagerank - top_grado))

## Tarea 3 — Detección de comunidades (Louvain)

**Responsable:** Emerson Ballesteros (líder)

**Problema:** ¿qué grupos de direcciones interactúan mucho entre sí?
En Ethereum los exchanges y los DEX forman "ecosistemas": se relacionan mucho
entre ellos y poco con el resto.

**Equivalencia matemática:** maximizar la **modularidad** $Q$. Una partición
es buena si hay más aristas *dentro* de los grupos de las que habría en un
grafo aleatorio:

$$Q = \frac{1}{2m}\sum_{i,j}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i, c_j)$$

- $A_{ij}$: peso de la arista entre $i$ y $j$.
- $k_i$: grado del nodo $i$.
- $m$: suma total de pesos.
- $\delta(c_i,c_j)$: 1 si $i$ y $j$ están en el mismo grupo, 0 si no.

Valores de $Q > 0.3$ indican una buena estructura de comunidades.

**Alternativas consideradas:**

| Algoritmo | ¿Por qué sí / no? |
|-----------|-------------------|
| Girvan-Newman | Muy lento para grafos grandes ($O(V \cdot E^2)$) |
| Clustering espectral | Necesita álgebra lineal densa |
| Propagación de etiquetas | Rápido pero inestable |
| **Louvain** | Rápido y da buenos resultados |

**Elegimos Louvain**, que se resume en dos pasos:

1. Cada nodo empieza en su propia comunidad.
2. Repetimos: mover cada nodo a la comunidad vecina que más aumente $Q$,
   hasta que ya nada mejore.

In [ ]:
# Tarea 3 — usa el mismo `grafo` cargado una sola vez.
from src.comunidades import (
    desde_aristas,
    louvain_desde_aristas,
    modularidad,
    resumen_comunidades,
)

adj_no_dirigido = desde_aristas(grafo.aristas)
comunidades = louvain_desde_aristas(grafo.aristas, semilla=SEMILLA)
Q = modularidad(adj_no_dirigido, comunidades)

print(f"Modularidad Q = {Q:.3f}")
print(f"Número de comunidades = {len(set(comunidades.values()))}\n")

grupos = resumen_comunidades(comunidades, grafo.etiquetas)
for i, (c, nombres) in enumerate(
    sorted(grupos.items(), key=lambda kv: -len(kv[1]))[:10], 1
):
    muestra = ", ".join(sorted(nombres)[:5])
    if len(nombres) > 5:
        muestra += ", ..."
    print(f"Comunidad {i}: {len(nombres):>4} nodos | {muestra}")

# Tamaño de cada comunidad.
tamanos = [len(nombres) for nombres in grupos.values()]
plt.figure(figsize=(6, 4))
plt.bar(range(1, len(tamanos) + 1), tamanos, color="mediumpurple", edgecolor="white")
plt.xlabel("Comunidad")
plt.ylabel("Número de nodos")
plt.title(f"Tamaño de las comunidades (Q = {Q:.3f})")
plt.show()

In [ ]:
# Experimento T3: Louvain es estocástico. Probamos varias semillas y
# comparamos la modularidad Q y el número de comunidades.
from src.comunidades import louvain

semillas = [1, 7, 42, 123, 2024]
resultados = []

for s in semillas:
    c = louvain(adj_no_dirigido, semilla=s)
    resultados.append((s, modularidad(adj_no_dirigido, c), len(set(c.values()))))

print(f"{'Semilla':>8} {'Q':>8} {'Comunidades':>12}")
print("-" * 30)
for s, q, n in resultados:
    print(f"{s:>8} {q:>8.3f} {n:>12}")

mejor = max(resultados, key=lambda x: x[1])
print(f"\nMejor semilla: {mejor[0]} con Q = {mejor[1]:.3f}")

plt.figure(figsize=(6, 4))
plt.bar([str(s) for s, _, _ in resultados], [q for _, q, _ in resultados],
        color="seagreen")
plt.xlabel("Semilla")
plt.ylabel("Modularidad Q")
plt.title("Sensibilidad de Louvain a la semilla")
plt.ylim(0, 1)
plt.show()

## Tarea 4 — Árbol de expansión mínima (MST)

**Responsabilidad:** grupal

**Problema:** ¿cuál es el conjunto mínimo de conexiones que mantiene a toda
la red conectada?

**Equivalencia matemática:** árbol de expansión mínima. Es un árbol (sin
ciclos) que toca todos los nodos con el menor costo total.

**Alternativas consideradas:**

| Algoritmo | ¿Por qué sí / no? |
|-----------|-------------------|
| Prim | Necesita una cola de prioridad |
| **Kruskal** | Ordena aristas y usa Union-Find |

**Elegimos Kruskal** porque Union-Find es una estructura de datos clásica que
vale la pena estudiar.


In [ ]:
# Tarea 4 — usa el mismo `grafo` cargado una sola vez.
from src.arbol_expansion import kruskal_desde_aristas, resumen_mst

mst, costo = kruskal_desde_aristas(grafo.aristas)
resumen = resumen_mst(mst)

print(f"MST: {resumen['aristas']} aristas, costo total = {resumen['costo_total']:.1f}\n")

print("Primeras aristas del árbol de expansión mínima:")
for a, b, w in sorted(mst, key=lambda e: e[2])[:20]:
    print(f"  {grafo.nombre(a):<16} -- {grafo.nombre(b):<16} (peso {w:.1f})")

In [ ]:
# Experimento T4: comparar el costo del MST con el peso total de la red.
peso_total = sum(peso for _, _, peso in grafo.aristas)
fraccion = costo / peso_total if peso_total else 0.0

print(f"Peso total de las aristas : {peso_total:,.1f}")
print(f"Costo del MST             : {costo:,.1f}")
print(f"Fracción del peso total   : {fraccion:.2%}")

# Distribución de los pesos de las aristas del MST.
pesos_mst = sorted(w for _, _, w in mst)
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(pesos_mst) + 1), pesos_mst, "o-", markersize=3,
         color="darkorange")
plt.xlabel("Arista del MST (ordenada por peso)")
plt.ylabel("Peso")
plt.title("Pesos de las aristas del MST")
plt.grid(True, alpha=0.3)
plt.show()

## 2. Análisis del dataset completo (38,9 M)

Esta sección **no usa el objeto `grafo`** de la Sección 1. Consulta el
**Parquet completo** (~38,9 millones de transferencias, ~0,8 GB) directamente
con DuckDB, sin cargarlo entero en memoria.

Aquí está el análisis extenso:

1. **Estadísticas y distribuciones** del dataset completo.
2. **Estudio de escalabilidad**: subgrafos conectados reales de tamaño creciente y
   tiempo de cada algoritmo.

> Los montos están en **unidades mínimas de cada token** (los decimales
> dependen del contrato). No se suman como USD entre tokens distintos; por eso
> el ranking principal usa el **número de transferencias**.

Si el Parquet no está disponible, la sección se omite y el resto del notebook
sigue funcionando.

### 2.0 Configuración

Cambia estos valores para agrandar o achicar el análisis.

In [ ]:
# ============ CONFIGURACIÓN DEL ANÁLISIS EXTENSO ============
# Todo se calcula con el dataset real (el Parquet). No hay datos inventados.

import os

# Ruta al Parquet completo (~0,8 GB).
RUTA_PARQUET = "/content/transferencias.parquet"
RUTA_PARQUET_DRIVE = "/content/drive/MyDrive/ED2/transferencias.parquet"
RUTA_PARQUET_LOCAL = "data/procesado/transferencias.parquet"

for _ruta in (RUTA_PARQUET, RUTA_PARQUET_DRIVE, RUTA_PARQUET_LOCAL):
    if os.path.isfile(_ruta):
        RUTA_PARQUET = _ruta
        break

# Cuántas filas usar en el análisis descriptivo (None = todas).
FILAS_ANALISIS = None

# Tamaños del benchmark (número de aristas reales extraídas del dataset).
TAMANOS = [1_000, 5_000, 10_000, 25_000, 50_000, 100_000, 250_000, 500_000]

# Repeticiones por medición (se usa la mediana) y tiempo máximo por corrida.
REPETICIONES = 3
TIMEOUT_SEG = 30

# Ajustes de DuckDB para no agotar la RAM (Colab o portátil).
MEMORIA_DUCKDB = "2GB"
THREADS_DUCKDB = 2

# Opcional: analizar un solo token (dirección del contrato).
TOKEN = None

print("Parquet:", RUTA_PARQUET)
print("¿Existe?", os.path.isfile(RUTA_PARQUET))

### 2.1 Estadísticas y distribuciones del dataset completo

In [ ]:
import os

if RUTA_PARQUET and os.path.isfile(RUTA_PARQUET):
    try:
        import duckdb  # noqa: F401
    except ImportError:
        import subprocess
        import sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "pyarrow"], check=True)

    from src.analisis import abrir, estadisticas

    con = abrir(memoria=MEMORIA_DUCKDB, threads=THREADS_DUCKDB)
    datos = estadisticas(con, RUTA_PARQUET, filas=FILAS_ANALISIS, token=TOKEN)
    r = datos["resumen"]

    print(f"Transferencias analizadas : {r['transferencias']:,}")
    print(f"Direcciones origen        : {r['origenes']:,}")
    print(f"Direcciones destino       : {r['destinos']:,}")
    print(f"Volumen total (crudo)     : {r['volumen']:,.0f}")
    print(f"Valor promedio (crudo)    : {r['promedio']:,.2f}")
    print(f"Valor máximo (crudo)      : {r['maximo']:,.2f}")

    # Distribución de montos en escala logarítmica.
    bins = [str(int(b)) for b, _ in datos["valores"]]
    conteos = [n for _, n in datos["valores"]]
    plt.figure(figsize=(7, 4))
    plt.bar(bins, conteos, color="steelblue")
    plt.xlabel("log10(valor)")
    plt.ylabel("Número de transferencias")
    plt.title("Distribución de montos (dataset completo)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # Distribución de grados en escala log-log.
    gx = [g for g, _ in datos["grados"] if g > 0]
    gy = [n for g, n in datos["grados"] if g > 0]
    plt.figure(figsize=(7, 4))
    plt.loglog(gx, gy, "o-", markersize=3, color="darkorange")
    plt.xlabel("Grado (conexiones)")
    plt.ylabel("Número de direcciones")
    plt.title("Distribución de grados (log-log)")
    plt.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Actividad por día (si el dataset trae fecha o número de bloque).
    if datos["actividad"]:
        dias = [d for d, _ in datos["actividad"]]
        n_dias = [n for _, n in datos["actividad"]]
        plt.figure(figsize=(9, 4))
        plt.plot(dias, n_dias, color="seagreen")
        plt.xlabel("Día")
        plt.ylabel("Transferencias")
        plt.title("Actividad por día")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

    # Top 10 direcciones por número de transferencias (comparable entre tokens).
    etiquetas_ex = {}
    ruta_etiquetas = "data/labels/exchange_labels.csv"
    if os.path.isfile(ruta_etiquetas):
        import csv
        with open(ruta_etiquetas, newline="", encoding="utf-8") as fh:
            for fila in csv.DictReader(fh):
                etiquetas_ex[fila["address"].lower()] = fila["name"]

    top = datos["top_actividad"][:10]
    nombres = [etiquetas_ex.get(a, a[:10] + "...") for a, _ in top]
    valores = [n for _, n in top]
    plt.figure(figsize=(7, 4))
    plt.barh(nombres[::-1], valores[::-1], color="mediumpurple")
    plt.xlabel("Número de transferencias")
    plt.title("Top 10 direcciones por actividad")
    plt.tight_layout()
    plt.show()

    # Distribución por token (si existe la columna).
    if datos["tokens"]:
        top_tokens = datos["tokens"][:10]
        etiquetas_t = [t[:10] + "..." for t, _ in top_tokens]
        valores_t = [n for _, n in top_tokens]
        plt.figure(figsize=(7, 4))
        plt.barh(etiquetas_t[::-1], valores_t[::-1], color="teal")
        plt.xlabel("Número de transferencias")
        plt.title("Top 10 tokens por número de transferencias")
        plt.tight_layout()
        plt.show()
else:
    print("Sección omitida: configura RUTA_PARQUET con la ruta del Parquet.")
    print("Genera el Parquet con notebooks/Preparar_Dataset_Colab.ipynb.")

### 2.2 Estudio de escalabilidad (benchmark)

Extraemos subgrafos conectados reales de tamaño creciente y medimos el tiempo de cada
algoritmo. Si uno se pasa de `TIMEOUT_SEG`, se marca como *timeout* y no se
vuelve a probar en tamaños mayores (su costo solo puede crecer).

In [ ]:
import os

if RUTA_PARQUET and os.path.isfile(RUTA_PARQUET):
    from src.analisis import abrir, extraer_base, benchmark

    con = abrir(memoria=MEMORIA_DUCKDB, threads=THREADS_DUCKDB)
    base = extraer_base(con, RUTA_PARQUET, max(TAMANOS), token=TOKEN)
    resultados = benchmark(base, TAMANOS, repeticiones=REPETICIONES, timeout=TIMEOUT_SEG)

    print(f"\n{'Algoritmo':<14} {'Aristas':>10} {'Nodos':>8} {'Tiempo (s)':>12}")
    for fila in resultados:
        t = "timeout" if fila["tiempo"] is None else f"{fila['tiempo']:.4f}"
        print(f"{fila['algoritmo']:<14} {fila['aristas']:>10,} {fila['nodos']:>8,} {t:>12}")

    plt.figure(figsize=(8, 5))
    for nombre in ["Kruskal", "PageRank", "Bellman-Ford", "Louvain"]:
        puntos = [(f["aristas"], f["tiempo"]) for f in resultados
                  if f["algoritmo"] == nombre and f["tiempo"] is not None]
        if puntos:
            plt.loglog([p[0] for p in puntos], [p[1] for p in puntos], "o-", label=nombre)
    plt.xlabel("Aristas del subgrafo real")
    plt.ylabel("Tiempo (segundos)")
    plt.title("Escalabilidad de los algoritmos (datos reales)")
    plt.grid(True, which="both", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Sección omitida: configura RUTA_PARQUET con la ruta del Parquet.")

### 2.3 Cómo leer estos resultados

- **Kruskal** y **PageRank** crecen suave: aguantan subgrafos grandes.
- **Bellman-Ford** y **Louvain** se disparan: por eso se ejecutan sobre un
  subgrafo de miles de aristas.
- Los puntos que no aparecen en la gráfica se pasaron del `TIMEOUT_SEG`.

Esta comparación es la misma lección del burbuja vs. ordenamiento eficiente,
pero con algoritmos de grafos y sobre datos reales.


## Conclusiones

**Responsable:** Emerson Ballesteros

1. **Un mismo dataset admite varias preguntas.** Bellman-Ford responde
   caminos, PageRank responde importancia, Louvain responde grupos y Kruskal
   responde conectividad mínima.

2. **Las estructuras de datos importan.** La lista de adyacencia recorre el
   grafo, Union-Find detecta ciclos en Kruskal, y los diccionarios y conjuntos
   hacen que todo sea simple y legible en Python.

3. **La red no es un solo bloque.** Louvain encuentra comunidades claras
   (ecosistemas de DEX y de exchanges) separadas por puentes débiles.

4. **La escala importa.** El análisis del dataset completo (38,9 M) se hace
   con DuckDB; los algoritmos en Python puro se ejecutan sobre el mayor
   subgrafo viable, y el benchmark muestra dónde se rompe cada uno.

5. **Una sola carga, varias preguntas.** El mismo objeto `grafo` alimenta las
   cuatro tareas; ninguna vuelve a leer el dataset.

En resumen, lo valioso no fue memorizar algoritmos, sino entender **qué
estructura de datos usa cada uno, por qué funciona y hasta dónde escala**.